# IEEE fraud model training: XGBoost hyperparameter tuning

Same train, validation, and test construction as `02_5_model_training_balancing_tests.ipynb`. The training set is cut once to `MAX_TRAIN_ROWS` with a stratified sample, so the fraud rate stays about 3.5%. Validation and test stay at the natural fraud rate. There is no resampling and no `scale_pos_weight`.

Optuna searches XGBoost on the columns in `TOP_40_FEATURES`. Edit that table in `src/model_training/hyperparameter_tuning.py`, or replace `FEATURE_TABLE` in the feature cell below. Each row is `[rank, feature name, gain]`. Only the feature name is trained on.

The search maximizes validation PR-AUC. Early stopping still watches validation logloss and keeps the best validation round. The test set is scored once, after the search, on the best parameters. The F1 threshold is chosen on the natural validation set.

Tracking goes to the Docker MLflow server (`http://localhost:5000`). The final model is logged to `ieee_fraud_hyperparameter_tuning`, separate from the balancing comparison.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "runs_entrypoints":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import mlflow
import pandas as pd

from src.model_training.hyperparameter_tuning import (
    TOP_40_FEATURES,
    feature_names,
    tune_xgboost,
)
from src.model_training.training import ModelTrainingWorkflow
from src.run_time_configuration import BaseConfigParams

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TARGET_COLUMN = "is_fraud"
SEED = 42
MAX_TRAIN_ROWS = 100_000
N_TRIALS = 500
SEARCH_TIMEOUT_SECONDS = 600
EARLY_STOPPING_ROUNDS = 5
EARLY_STOPPING_MIN_DELTA = 0.001
DROP_FROM_FEATURES = ["transaction_id", "seconds_from_reference", TARGET_COLUMN]


In [2]:
def _load_env(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())


_load_env(PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = "http://localhost:5000"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")


Project root: /home/martinezyamamotoarthur/modeling_fraud_system
Processed dir: /home/martinezyamamotoarthur/modeling_fraud_system/data/processed
MLflow tracking URI: http://localhost:5000


In [3]:
df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_validation = pd.read_parquet(PROCESSED_DIR / "validation.parquet")
df_test = pd.read_parquet(PROCESSED_DIR / "test.parquet")

for name, frame in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    print(name, frame.shape, "pos_rate", round(float(frame[TARGET_COLUMN].mean()), 4))


train (331633, 332) pos_rate 0.0352
validation (82909, 332) pos_rate 0.0352
test (175998, 332) pos_rate 0.0344


In [4]:
from sklearn.model_selection import train_test_split


def class_counts(df: pd.DataFrame) -> pd.Series:
    return df[TARGET_COLUMN].value_counts().sort_index()


def limit_training_rows(df: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    if len(df) <= max_rows:
        return df.reset_index(drop=True)
    kept, _ = train_test_split(
        df,
        train_size=max_rows,
        random_state=seed,
        stratify=df[TARGET_COLUMN],
    )
    return kept.reset_index(drop=True)


df_train_capped = limit_training_rows(df_train, MAX_TRAIN_ROWS, SEED)

print("train loaded", df_train.shape)
print(class_counts(df_train))
print("train capped", df_train_capped.shape)
print(class_counts(df_train_capped))
print("validation", df_validation.shape)
print(class_counts(df_validation))
print("test", df_test.shape)
print(class_counts(df_test))

del df_train


train loaded (331633, 332)
is_fraud
0    319953
1     11680
Name: count, dtype: int64
train capped (100000, 332)
is_fraud
0    96478
1     3522
Name: count, dtype: int64
validation (82909, 332)
is_fraud
0    79989
1     2920
Name: count, dtype: int64
test (175998, 332)
is_fraud
0    169935
1      6063
Name: count, dtype: int64


## Features

Edit `FEATURE_TABLE` to change the columns. Leave it as `TOP_40_FEATURES` to use the table in `src/model_training/hyperparameter_tuning.py`. That default is the gain ranking from the no-balancing run. Add, remove, or reorder rows. Each row is `[rank, feature name, gain]`. Rank and gain are notes; training uses the name only.


In [5]:
# Edit this assignment to train on a different column list.
FEATURE_TABLE = TOP_40_FEATURES
SELECTED_FEATURES = feature_names(FEATURE_TABLE)

print(f"{len(SELECTED_FEATURES)} features")
for rank, name, gain in FEATURE_TABLE:
    print(f"{int(rank):>2}  {name}  {gain}")


40 features
 1  card1_target_rate  2898.5080566406
 2  card1_x_product_channel_target_rate  2619.8493652344
 3  C2  1788.8270263672
 4  V217  1463.5167236328
 5  V279  1325.2569580078
 6  C11  1288.9261474609
 7  V29  1281.1726074219
 8  V283  1202.521484375
 9  V95  1136.4382324219
10  V312  1072.3162841797
11  C6  985.2222290039
12  V282  894.4429321289
13  C12  891.3984375
14  V258  871.4403686523
15  C14  862.3701782227
16  card_D10_std  859.451965332
17  V313  833.7211914062
18  card_C9_mean  800.0765991211
19  V57  793.2509155273
20  flag_outsider15  753.5994262695
21  V69  727.9138793945
22  V90  709.7984008789
23  D3  705.2971801758
24  V53  701.7498168945
25  C1  693.0908813477
26  card_C13_nunique  680.6305541992
27  V265  663.9107666016
28  V314  645.0280761719
29  amount_cents  589.6865234375
30  D2  581.2379150391
31  V35  529.9074707031
32  card_V309_nunique  527.7366943359
33  V285  516.6747436523
34  C13  514.0857543945
35  V288  506.8506774902
36  purchaser_email_domai

## Fit the search data

No balancing. Training rows stay at the capped natural fraud rate. Validation and test are unchanged. Preprocessing keeps the selected columns and numeric values only, including native missing values.


In [6]:
def features_and_target(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    y = df[TARGET_COLUMN]
    x = df.drop(columns=[column for column in DROP_FROM_FEATURES if column in df.columns])
    return x, y


runtime_params = BaseConfigParams(
    experiment_name="ieee_fraud_hyperparameter_tuning",
    mlflow_run_name="xgb_top40_optuna",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)
workflow = ModelTrainingWorkflow(runtime_params)
workflow.df_train = df_train_capped
workflow.df_validation = df_validation
workflow.df_test = df_test
workflow.x_train, workflow.y_train = features_and_target(df_train_capped)
workflow.x_validation, workflow.y_validation = features_and_target(df_validation)
workflow.x_test, workflow.y_test = features_and_target(df_test)
workflow.generate_preprocessed_data(
    numeric_only=True,
    selected_features=SELECTED_FEATURES,
)

print("train", workflow.x_train_preproc.shape, "pos_rate", round(float(workflow.y_train.mean()), 4))
print("validation", workflow.x_validation_preproc.shape, "pos_rate", round(float(workflow.y_validation.mean()), 4))
print("test", workflow.x_test_preproc.shape, "pos_rate", round(float(workflow.y_test.mean()), 4))
print("columns", list(workflow.x_train_preproc.columns))


2026-10-11 17:52:04,718 - src.model_training.training - INFO - Initialized the SolvencyNewCustomersModelTraining workflow.
2026-10-11 17:52:04,725 - src.model_training.training - INFO - Generating preprocessed data by copying original data.
2026-10-11 17:52:04,726 - src.model_training.training - INFO - Using selected features: 40 features specified
2026-10-11 17:52:04,727 - src.model_training.training - INFO - Selected features: ['card1_target_rate', 'card1_x_product_channel_target_rate', 'C2', 'V217', 'V279', 'C11', 'V29', 'V283', 'V95', 'V312', 'C6', 'V282', 'C12', 'V258', 'C14', 'card_D10_std', 'V313', 'card_C9_mean', 'V57', 'flag_outsider15', 'V69', 'V90', 'D3', 'V53', 'C1', 'card_C13_nunique', 'V265', 'V314', 'amount_cents', 'D2', 'V35', 'card_V309_nunique', 'V285', 'C13', 'V288', 'purchaser_email_domain_target_rate', 'V309', 'V221', 'V284', 'V131']
2026-10-11 17:52:04,764 - src.model_training.training - INFO - Filtering to numeric columns: 40 out of 40 total columns
2026-10-11 17

train (100000, 40) pos_rate 0.0352
validation (82909, 40) pos_rate 0.0352
test (175998, 40) pos_rate 0.0344
columns ['card1_target_rate', 'card1_x_product_channel_target_rate', 'C2', 'V217', 'V279', 'C11', 'V29', 'V283', 'V95', 'V312', 'C6', 'V282', 'C12', 'V258', 'C14', 'card_D10_std', 'V313', 'card_C9_mean', 'V57', 'flag_outsider15', 'V69', 'V90', 'D3', 'V53', 'C1', 'card_C13_nunique', 'V265', 'V314', 'amount_cents', 'D2', 'V35', 'card_V309_nunique', 'V285', 'C13', 'V288', 'purchaser_email_domain_target_rate', 'V309', 'V221', 'V284', 'V131']


## Search

Each trial trains XGBoost with early stopping on the validation set. Optuna keeps the trial with the highest validation PR-AUC. The search stops at `N_TRIALS` or after `SEARCH_TIMEOUT_SECONDS`, whichever comes first. `EARLY_STOPPING_ROUNDS` and `EARLY_STOPPING_MIN_DELTA` are set in the first cell.


In [7]:
study = tune_xgboost(
    workflow,
    n_trials=N_TRIALS,
    seed=SEED,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    timeout=SEARCH_TIMEOUT_SECONDS,
)

best_hyperparameters = study.best_trial.user_attrs["hyperparameters"]
print("best validation PR-AUC", round(study.best_value, 4))
print("best iteration", study.best_trial.user_attrs["best_iteration"])
print(best_hyperparameters)

trials = study.trials_dataframe().sort_values("value", ascending=False)
trials.head(10)


2026-10-11 17:52:04,806 - src.model_training.hyperparameter_tuning - INFO - Starting XGBoost Optuna search: 30 trials, 82909 rows, 40 features.
[I 2026-10-11 17:52:04,808] A new study created in memory with name: xgboost_no_balancing


  0%|          | 0/30 [00:00<?, ?it/s]

/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:52:05] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
2026-10-11 17:52:06,015 - src.model_training.hyperparameter_tuning - INFO - Trial 0 validation PR-AUC 0.6256 best_iteration 35
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:52:06] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[I 2026-10-11 17:52:06,016] Trial 0 finished with value: 0.6256248638178267 and parameters: {'n_estimators': 218, 'max_depth': 8, 'learning_rate': 0.08960785365368121, 'min_child_weight': 10.401663679887308, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.5779972601681014, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.2537815508265665}. Best is trial 0 with value: 0.6256248638178267.


2026-10-11 17:52:06,892 - src.model_training.hyperparameter_tuning - INFO - Trial 1 validation PR-AUC 0.5672 best_iteration 32
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:52:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


...

2026-10-11 17:52:37,665 - src.model_training.hyperparameter_tuning - INFO - Trial 28 validation PR-AUC 0.6363 best_iteration 29
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:52:37] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[I 2026-10-11 17:52:37,666] Trial 28 finished with value: 0.6362863401629776 and parameters: {'n_estimators': 77, 'max_depth': 7, 'learning_rate': 0.1407156706631352, 'min_child_weight': 1.5098619097572192, 'subsample': 0.6102825872509905, 'colsample_bytree': 0.5989681897691307, 'gamma': 0.7929079529565294, 'reg_alpha': 0.28782025319807697, 'reg_lambda': 0.07774385968286196}. Best is trial 10 with value: 0.651204716297015.


2026-10-11 17:52:38,520 - src.model_training.hyperparameter_tuning - INFO - Trial 29 validation PR-AUC 0.6428 best_iteration 28
2026-10-11 17:52:38,523 - src.model_training.hyperparameter_tuning - INFO - Optuna search finished. Best validation PR-AUC 0.6512 with {'n_estimators': 118, 'max_depth': 8, 'learning_rate': 0.12988182243767124, 'min_child_weight': 1.029965479668185, 'subsample': 0.7090747508804338, 'colsample_bytree': 0.5347932177377853, 'gamma': 0.4707107273372464, 'reg_alpha': 0.014414146389086174, 'reg_lambda': 0.35238450522799947, 'n_jobs': 4}.


[I 2026-10-11 17:52:38,521] Trial 29 finished with value: 0.642779077899193 and parameters: {'n_estimators': 162, 'max_depth': 8, 'learning_rate': 0.13771531088494524, 'min_child_weight': 1.428871831630709, 'subsample': 0.7164413217393549, 'colsample_bytree': 0.6333964936949484, 'gamma': 1.7681550976261136, 'reg_alpha': 0.09748777135388093, 'reg_lambda': 0.38660852530949474}. Best is trial 10 with value: 0.651204716297015.
best validation PR-AUC 0.6512
best iteration 30
{'n_estimators': 118, 'max_depth': 8, 'learning_rate': 0.12988182243767124, 'min_child_weight': 1.029965479668185, 'subsample': 0.7090747508804338, 'colsample_bytree': 0.5347932177377853, 'gamma': 0.4707107273372464, 'reg_alpha': 0.014414146389086174, 'reg_lambda': 0.35238450522799947, 'n_jobs': 4}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,user_attrs_best_iteration,user_attrs_hyperparameters,user_attrs_val_pr_auc,system_attrs_tpe:relative_params:0,state
10,10,0.651205,2026-10-11 17:52:18.277004,2026-10-11 17:52:19.291364,0 days 00:00:01.014360,0.534793,0.470711,0.129882,8,1.029965,118,0.014414,0.352385,0.709075,30,"{'n_estimators': 118, 'max_depth': 8, 'learnin...",0.651205,"{""colsample_bytree"": 0.5347932177377853, ""gamm...",COMPLETE
21,21,0.648345,2026-10-11 17:52:29.462740,2026-10-11 17:52:30.496063,0 days 00:00:01.033323,0.547807,0.709266,0.092277,8,1.007957,98,0.097078,0.120029,0.615845,34,"{'n_estimators': 98, 'max_depth': 8, 'learning...",0.648345,"{""colsample_bytree"": 0.5478074003424516, ""gamm...",COMPLETE
23,23,0.647303,2026-10-11 17:52:31.441721,2026-10-11 17:52:32.443539,0 days 00:00:01.001818,0.551869,0.531045,0.121063,8,1.152408,193,0.008552,0.129249,0.650437,29,"{'n_estimators': 193, 'max_depth': 8, 'learnin...",0.647303,"{""colsample_bytree"": 0.5518693963671534, ""gamm...",COMPLETE
11,11,0.646431,2026-10-11 17:52:19.293188,2026-10-11 17:52:20.276087,0 days 00:00:00.982899,0.623237,1.242006,0.101703,8,1.685408,185,0.441973,0.110248,0.656341,33,"{'n_estimators': 185, 'max_depth': 8, 'learnin...",0.646431,"{""colsample_bytree"": 0.6232372788938783, ""gamm...",COMPLETE
27,27,0.643553,2026-10-11 17:52:35.448531,2026-10-11 17:52:36.785891,0 days 00:00:01.337360,0.526147,0.216033,0.057716,8,2.047177,171,0.022025,0.918635,0.632362,48,"{'n_estimators': 171, 'max_depth': 8, 'learnin...",0.643553,"{""colsample_bytree"": 0.5261473991168374, ""gamm...",COMPLETE
24,24,0.643395,2026-10-11 17:52:32.445684,2026-10-11 17:52:33.511821,0 days 00:00:01.066137,0.559156,1.228496,0.107532,8,1.626519,278,0.001547,0.676806,0.651582,34,"{'n_estimators': 278, 'max_depth': 8, 'learnin...",0.643395,"{""colsample_bytree"": 0.5591558221329008, ""gamm...",COMPLETE
29,29,0.642779,2026-10-11 17:52:37.669005,2026-10-11 17:52:38.521490,0 days 00:00:00.852485,0.633396,1.768155,0.137715,8,1.428872,162,0.097488,0.386609,0.716441,28,"{'n_estimators': 162, 'max_depth': 8, 'learnin...",0.642779,"{""colsample_bytree"": 0.6333964936949484, ""gamm...",COMPLETE
16,16,0.642396,2026-10-11 17:52:24.325009,2026-10-11 17:52:25.457733,0 days 00:00:01.132724,0.603000,1.175069,0.066874,8,2.888508,263,0.127764,0.122118,0.640044,39,"{'n_estimators': 263, 'max_depth': 8, 'learnin...",0.642396,"{""colsample_bytree"": 0.6030002973512796, ""gamm...",COMPLETE
13,13,0.641982,2026-10-11 17:52:21.226815,2026-10-11 17:52:22.114382,0 days 00:00:00.887567,0.721669,1.230444,0.145983,7,1.134824,93,0.013103,0.886401,0.671388,30,"{'n_estimators': 93, 'max_depth': 7, 'learning...",0.641982,"{""colsample_bytree"": 0.7216689322249117, ""gamm...",COMPLETE
12,12,0.639937,2026-10-11 17:52:20.277997,2026-10-11 17:52:21.224892,0 days 00:00:00.946895,0.573812,0.738568,0.078408,7,1.423840,120,0.000053,0.041993,0.707089,33,"{'n_estimators': 120, 'max_depth': 7, 'learnin...",0.639937,"{""colsample_bytree"": 0.5738120402599283, ""gamm...",COMPLETE


## Train the best parameters

One final fit, logged to MLflow. Test metrics are computed here and were not used to pick the trial. The kept model is the best validation round.


In [8]:
results = workflow.train_model(
    hyperparameters=best_hyperparameters,
    model_type="xgboost",
    log_into_mlflow=True,
    show_plots=False,
    verbose=False,
    seed=SEED,
    run_name="xgb_top40_optuna",
    top_n_features=len(SELECTED_FEATURES),
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)
test_metrics = results["test_metrics"]
summary = {
    "run_id": workflow.run_time_config.mlflow_run_id,
    "n_features": len(SELECTED_FEATURES),
    "train_rows": int(len(workflow.y_train)),
    "train_pos_rate": float(workflow.y_train.mean()),
    "best_iteration": results["best_iteration"],
    "threshold": float(results["threshold"]),
    "precision": test_metrics["precision"],
    "recall": test_metrics["recall"],
    "f1": test_metrics["f1"],
    "pr_auc": test_metrics["pr_auc"],
    "auc": test_metrics["auc"],
    "ks_statistic": test_metrics["ks_statistic"],
    "optuna_val_pr_auc": study.best_value,
}
summary


2026-10-11 17:52:38,597 - src.model_training.training - INFO - Starting model training without hyperparameter tuning using xgboost.
2026-10-11 17:52:38,638 - src.model_training.training - INFO - Training xgboost model with fixed hyperparameters and early stopping (patience 5, minimum validation improvement 0.001).
/home/martinezyamamotoarthur/modeling_fraud_system/.venv/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [17:52:38] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()
2026-10-11 17:52:39,558 - src.model_training.training - INFO - Best iteration on the validation set: 30.
2026-10-11 17:52:39,592 - src.model_training.training - INFO - Calculated optimal threshold (F1-based) on validation set: 0.2914
2026-10-11 17:52:39,785 - src.model_training.evaluation - INFO - Training Metrics - AUC: 0.9574, PR-AUC: 0.7532, KS: 0.7738
2026-10-11 17:52:39,871 - src.model_tr

🏃 View run xgb_top40_optuna at: http://localhost:5000/#/experiments/155/runs/867ae72c14ff4acdb41d3071d501e6fe
🧪 View experiment at: http://localhost:5000/#/experiments/155


{'run_id': '867ae72c14ff4acdb41d3071d501e6fe',
 'n_features': 40,
 'train_rows': 100000,
 'train_pos_rate': 0.03522,
 'best_iteration': 30,
 'threshold': 0.291440486907959,
 'precision': 0.6110813704496788,
 'recall': 0.37654626422563087,
 'f1': 0.4659659148892744,
 'pr_auc': 0.45738210691490844,
 'auc': 0.8748755620733624,
 'ks_statistic': np.float32(0.59967095),
 'optuna_val_pr_auc': 0.651204716297015}